# 🎵 Spotify Data Analysis
**Author:** Dheeraj Kandpal  
**Dataset:** 1,200 tracks · 20 artists · 2018–2023  
**Tools:** Python · pandas · matplotlib · seaborn

---
### Objective
Analyse Spotify streaming data to answer real business questions:
- Which artists and genres dominate streams?
- How do audio features (danceability, energy, valence) relate to popularity?
- What seasonal and yearly trends exist in listening behaviour?
- What mood profiles do different genres carry?


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Spotify dark theme ──────────────────────────────────────────────────────
BG, SURFACE = '#0D1117', '#161B22'
GREEN, WHITE, MUTED = '#1DB954', '#E6EDF3', '#8B949E'
ACCENT, CORAL, AMBER = '#58A6FF', '#FF6B6B', '#F0A500'
PALETTE = [GREEN, ACCENT, CORAL, AMBER, '#B79FFF',
           '#56D364', '#FF8C42', '#79C0FF', '#FFA8A8', '#D2A8FF']

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   SURFACE,
    'text.color':       WHITE,
    'xtick.color':      MUTED,
    'ytick.color':      MUTED,
})

df = pd.read_csv('../data/spotify_tracks.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Exploratory Overview

In [ ]:
print('=== DATASET SUMMARY ===')
print(f'Total tracks    : {len(df):,}')
print(f'Unique artists  : {df["artist_name"].nunique()}')
print(f'Unique genres   : {df["genre"].nunique()}')
print(f'Year range      : {df["release_year"].min()} – {df["release_year"].max()}')
print(f'Total streams   : {df["streams"].sum():,.0f}')
print(f'Avg popularity  : {df["popularity"].mean():.1f} / 100')
print(f'Avg duration    : {df["duration_min"].mean():.1f} min')
print()
print('=== NULL CHECK ===')
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().any() else 'No null values ✓')

In [ ]:
# Descriptive statistics for numeric features
df[['streams','popularity','danceability','energy',
    'valence','tempo','acousticness']].describe().round(3)

## 3. Top Artists by Total Streams

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5), facecolor=BG)
top = (df.groupby('artist_name')['streams'].sum()
         .sort_values(ascending=True).tail(10))
bars = ax.barh(top.index, top.values / 1e6, color=PALETTE[:10][::-1], height=0.6)
for bar, val in zip(bars, top.values):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.0f}M', va='center', color=WHITE, fontsize=8.5)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='x', color='#21262D', linewidth=0.6, linestyle='--')
ax.set_xlabel('Total Streams (Millions)', color=MUTED)
ax.set_title('Top 10 Artists by Total Streams', color=WHITE,
             fontsize=13, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}M'))
ax.tick_params(axis='y', colors=WHITE)
plt.tight_layout()
plt.savefig('../charts/01_top_artists_streams.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()

**Insight:** Taylor Swift leads with **504M streams** followed by Billie Eilish (443M) and Kendrick Lamar (415M).
Pop dominates the top spots, but Bollywood artists (Vishal-Shekhar) hold strong in the top 10.


## 4. Streams by Genre

In [ ]:
genre_streams = (df.groupby('genre')['streams'].sum()
                   .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
bars = ax.bar(genre_streams.index, genre_streams.values / 1e6,
              color=[GREEN if i==0 else ACCENT if i==1 else MUTED
                     for i in range(len(genre_streams))],
              width=0.55, edgecolor='none')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{bar.get_height():.0f}M', ha='center', color=MUTED, fontsize=8)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='y', color='#21262D', linewidth=0.6, linestyle='--')
ax.set_ylabel('Streams (Millions)', color=MUTED)
ax.set_title('Total Streams by Genre', color=WHITE, fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=30, ha='right', color=WHITE, fontsize=8.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}M'))
plt.tight_layout()
plt.savefig('../charts/02_streams_by_genre.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()

# Genre share table
total = genre_streams.sum()
share = (genre_streams / total * 100).round(1)
print('\nGenre share of total streams:')
for g, s in share.items():
    print(f'  {g:<20} {s}%')

## 5. Popularity Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), facecolor=BG)
ax.hist(df['popularity'], bins=30, color=GREEN, edgecolor=BG, linewidth=0.4, alpha=0.85)
ax.axvline(df['popularity'].mean(),   color=CORAL, lw=1.8, ls='--',
           label=f'Mean: {df["popularity"].mean():.1f}')
ax.axvline(df['popularity'].median(), color=AMBER, lw=1.8, ls=':',
           label=f'Median: {df["popularity"].median():.1f}')
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='y', color='#21262D', linewidth=0.6, linestyle='--')
ax.set_xlabel('Popularity Score', color=MUTED)
ax.set_ylabel('Number of Tracks', color=MUTED)
ax.set_title('Distribution of Track Popularity Scores',
             color=WHITE, fontsize=13, fontweight='bold', pad=12)
ax.legend(frameon=False, labelcolor=WHITE, fontsize=9)
plt.tight_layout()
plt.savefig('../charts/03_popularity_distribution.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()

## 6. Yearly Streaming Trends

In [ ]:
yearly = df.groupby('release_year')['streams'].sum() / 1e6

fig, ax = plt.subplots(figsize=(9, 4.5), facecolor=BG)
ax.plot(yearly.index, yearly.values, color=GREEN, linewidth=2.5,
        marker='o', markersize=7, markerfacecolor=WHITE, markeredgecolor=GREEN)
ax.fill_between(yearly.index, yearly.values, alpha=0.12, color=GREEN)
for x, y in zip(yearly.index, yearly.values):
    ax.text(x, y + 5, f'{y:.0f}M', ha='center', color=MUTED, fontsize=8.5)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='y', color='#21262D', linewidth=0.6, linestyle='--')
ax.set_xlabel('Year', color=MUTED)
ax.set_ylabel('Streams (Millions)', color=MUTED)
ax.set_title('Total Streams per Release Year',
             color=WHITE, fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(yearly.index)
plt.tight_layout()
plt.savefig('../charts/04_yearly_trends.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()

# YoY growth
yoy = yearly.pct_change() * 100
print('Year-over-Year growth:')
for yr, g in yoy.dropna().items():
    print(f'  {yr}: {g:+.1f}%')

## 7. Audio Features: Danceability vs Energy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
scatter = ax.scatter(df['danceability'], df['energy'],
                     c=df['popularity'], cmap='RdYlGn',
                     alpha=0.55, s=18, linewidths=0)
cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Popularity', color=MUTED, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=MUTED, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=MUTED)
cbar.outline.set_edgecolor('#21262D')
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(color='#21262D', linewidth=0.4, linestyle='--')
ax.set_xlabel('Danceability →', color=MUTED, fontsize=10)
ax.set_ylabel('Energy →', color=MUTED, fontsize=10)
ax.set_title('Danceability vs Energy (coloured by Popularity)',
             color=WHITE, fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../charts/05_danceability_vs_energy.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()

# Correlation
corr = df[['popularity','danceability','energy','valence',
           'acousticness','tempo','speechiness']].corr()['popularity'].drop('popularity')
print('Correlation with Popularity:')
for feat, val in corr.sort_values(ascending=False).items():
    print(f'  {feat:<15} {val:+.3f}')

## 8. Monthly Streams Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4), facecolor=BG)
pivot = df.pivot_table(index='release_year', columns='release_month',
                       values='streams', aggfunc='sum') / 1e6
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
sns.heatmap(pivot, ax=ax, cmap='YlGn', linewidths=0.4, linecolor=BG,
            annot=True, fmt='.0f', annot_kws={'size': 7.5, 'color': '#0D1117'},
            cbar_kws={'shrink': 0.8})
ax.set_xticklabels(month_labels, color=WHITE, fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), color=WHITE, fontsize=9, rotation=0)
ax.set_xlabel('Month', color=MUTED, fontsize=10)
ax.set_ylabel('Year',  color=MUTED, fontsize=10)
ax.set_title('Streams Heatmap — Year × Month (Millions)',
             color=WHITE, fontsize=13, fontweight='bold', pad=12)
for sp in ax.spines.values(): sp.set_visible(False)
ax.collections[0].colorbar.ax.tick_params(colors=MUTED, labelsize=8)
plt.tight_layout()
plt.savefig('../charts/06_monthly_heatmap.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()

## 9. Audio Feature Radar by Genre

In [ ]:
top5 = df.groupby('genre')['streams'].sum().nlargest(5).index.tolist()
feats = ['danceability','energy','valence','acousticness','speechiness']
angles = np.linspace(0, 2*np.pi, len(feats), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True), facecolor=BG)
ax.set_facecolor(SURFACE)
ax.spines['polar'].set_color('#21262D')
ax.grid(color='#21262D', linewidth=0.6)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(feats, color=WHITE, fontsize=9)
ax.set_yticks([])
colors_r = [GREEN, ACCENT, CORAL, AMBER, '#B79FFF']
for i, genre in enumerate(top5):
    vals = df[df['genre']==genre][feats].mean().tolist() + [None]
    vals[-1] = vals[0]
    ax.plot(angles, vals, color=colors_r[i], linewidth=2, label=genre)
    ax.fill(angles, vals, color=colors_r[i], alpha=0.08)
ax.set_title('Audio Feature Profile by Genre (Top 5)',
             color=WHITE, fontsize=13, fontweight='bold', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.15),
          frameon=False, labelcolor=WHITE, fontsize=9)
plt.tight_layout()
plt.savefig('../charts/07_audio_features_radar.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()

## 10. Mood Distribution by Genre

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), facecolor=BG)
mood_genre = (df[df['genre'].isin(top5)]
              .groupby(['genre','mood']).size().unstack(fill_value=0))
mood_genre.plot(kind='bar', ax=ax,
                color=PALETTE[:len(mood_genre.columns)],
                width=0.7, edgecolor='none')
ax.set_facecolor(SURFACE)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='y', color='#21262D', linewidth=0.6, linestyle='--')
ax.set_title('Mood Distribution Across Top 5 Genres',
             color=WHITE, fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Track Count', color=MUTED)
ax.set_xlabel('')
plt.xticks(rotation=15, ha='right', color=WHITE)
ax.legend(frameon=False, labelcolor=WHITE, fontsize=8.5,
          loc='upper right', ncol=2)
plt.tight_layout()
plt.savefig('../charts/08_mood_by_genre.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()

## 11. Key Findings Summary

| # | Finding | Implication |
|---|---------|-------------|
| 1 | **Pop** accounts for the highest total streams | Pop artists should be prioritised for playlist placements |
| 2 | **Danceability** has the strongest positive correlation with popularity | High-danceability tracks perform better in recommendations |
| 3 | **Sep–Oct** consistently show stream spikes across years | Festive season releases outperform other months |
| 4 | **Bollywood** dominates in track volume, not individual stream counts | Market is fragmented — no single breakout Bollywood track |
| 5 | Tracks with popularity **80–95** cluster in high-energy, high-danceability zones | Energy + danceability combo is a strong predictor of virality |
| 6 | **R&B and Hip-Hop** have the highest avg streams per track | Fewer tracks, higher impact — quality over quantity |


---
**Author:** Dheeraj Kandpal  
**GitHub:** [github.com/DheerajKandpal](https://github.com/DheerajKandpal)  
**Email:** dheeraj.kandpal@surepass.io
